In [3]:
import re
import json

# 1. The raw document with PII
document_text = """
Investment Agreement for Jane Smith (SSN: 987-65-4321, Account #AC-99120).
We guarantee Jane Smith an 18% annual return on investment without any market risk.
Contact: jane.smith@email.com, 555-123-4567.
"""

print("--- ORIGINAL DOCUMENT ---")
print(document_text)

# 2. PII Masking Service (Service 1)
pii_mapping = {}
masked_text = document_text

# Simple Regex patterns for the demo
ssn_pattern = r'\b\d{3}-\d{2}-\d{4}\b'
account_pattern = r'#AC-\d{5}'
name_pattern = r'\bJane Smith\b' # Hardcoded for demo speed, can use spaCy/Presidio in production

def mask_pii(text, mapping):
    # Mask SSN
    for match in re.finditer(ssn_pattern, text):
        token = f"[SSN_{len(mapping)+1}]"
        mapping[token] = match.group()
        text = text.replace(match.group(), token)

    # Mask Account
    for match in re.finditer(account_pattern, text):
        token = f"[ACCOUNT_{len(mapping)+1}]"
        mapping[token] = match.group()
        text = text.replace(match.group(), token)

    # Mask Name
    for match in re.finditer(name_pattern, text):
        token = f"[CLIENT_{len(mapping)+1}]"
        mapping[token] = match.group()
        text = text.replace(match.group(), token)

    return text

# Apply masking
masked_text = mask_pii(masked_text, pii_mapping)

print("\n--- MASKED TEXT (This is what gets sent to OpenAI/LLM) ---")
print(masked_text)

print("\n--- SECURE SERVER MAPPING (Stored in DB, never sent to LLM) ---")
print(json.dumps(pii_mapping, indent=2))

# 3. Dynamic Rehydration (Unmasking)
def rehydrate_text(text, mapping):
    for token, original in mapping.items():
        text = text.replace(token, original)
    return text

# Simulate the LLM returning a response using the masked tokens
llm_response = "The agreement for [CLIENT_1] is valid. SSN [SSN_1] verified."

print("\n--- RAW LLM RESPONSE (Masked) ---")
print(llm_response)

# Rehydrate for the UI
final_output = rehydrate_text(llm_response, pii_mapping)

print("\n--- FINAL UI OUTPUT (Rehydrated for the Officer) ---")
print(final_output)

--- ORIGINAL DOCUMENT ---

Investment Agreement for Jane Smith (SSN: 987-65-4321, Account #AC-99120).
We guarantee Jane Smith an 18% annual return on investment without any market risk.
Contact: jane.smith@email.com, 555-123-4567.


--- MASKED TEXT (This is what gets sent to OpenAI/LLM) ---

Investment Agreement for [CLIENT_3] (SSN: [SSN_1], Account [ACCOUNT_2]).
We guarantee [CLIENT_3] an 18% annual return on investment without any market risk.
Contact: jane.smith@email.com, 555-123-4567.


--- SECURE SERVER MAPPING (Stored in DB, never sent to LLM) ---
{
  "[SSN_1]": "987-65-4321",
  "[ACCOUNT_2]": "#AC-99120",
  "[CLIENT_3]": "Jane Smith",
  "[CLIENT_4]": "Jane Smith"
}

--- RAW LLM RESPONSE (Masked) ---
The agreement for [CLIENT_1] is valid. SSN [SSN_1] verified.

--- FINAL UI OUTPUT (Rehydrated for the Officer) ---
The agreement for [CLIENT_1] is valid. SSN 987-65-4321 verified.
